In [24]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.tree import plot_tree, DecisionTreeClassifier, DecisionTreeRegressor

In [25]:
df = pd.DataFrame()

lista = [
    r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202507Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202506Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202505Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202504Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202503Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202502Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202501Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202412Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202409Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202406Bens_Imoveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\202403Bens_Imoveis_Grupos.csv"
]
for i in lista:
    df1 = pd.read_csv(i, sep=';', encoding='latin1', dtype=str)
    df = pd.concat([df, df1], ignore_index=True)
df

,#Nome_da_Administradora,CNPJ_da_Administradora,Data_base,Código_do_grupo,Código_do_segmento,Número_da_assembléia_geral_ordinária,Valor_médio_do_bem,Índice_de_correção,Taxa_de_administração,Prazo_do_grupo_em_meses,Quantidade_de_cotas_ativas_em_dia,Quantidade_de_cotas_ativas_contempladas_inadimplentes,Quantidade_de_cotas_ativas_não_contempladas_inadimplentes,Quantidade_de_cotas_ativas_contempladas_no_mês,Quantidade_de_cotas_excluídas,Quantidade_de_cotas_ativas_quitadas,Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização
0,ITAÚ ADM DE CONSÓRCIOS LTDA ...,00000776,202507,00057,1,182,"232071,89",3,"20,60469",200,707,39,0,1,2967,397,90
1,ITAÚ ADM DE CONSÓRCIOS LTDA ...,00000776,202507,00058,1,179,"522509,37",3,"18,96488",200,608,43,1,8,2128,264,73
2,ITAÚ ADM DE CONSÓRCIOS LTDA ...,00000776,202507,00061,1,178,"198784,02",3,"21,30866",200,750,49,0,0,2620,398,79
3,ITAÚ ADM DE CONSÓRCIOS LTDA ...,00000776,202507,00062,1,177,"274389,27",3,"18,73604",200,718,32,2,5,2258,299,99
4,ITAÚ ADM DE CONSÓRCIOS LTDA ...,00000776,202507,00063,1,177,"237084,05",3,"10,91300",200,578,22,0,0,655,442,11
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25496,BANRISUL S.A. ADM CONSÓRCIOS ...,92692979,202403,10065,1,4,"150305,21",3,"18,68033",200,431,0,57,2,112,0,4
25497,BANRISUL S.A. ADM CONSÓRCIOS ...,92692979,202403,10066,1,4,"66466,51",3,"19,72265",200,357,0,36,1,103,0,3
25498,BANRISUL S.A. ADM CONSÓRCIOS ...,92692979,202403,10902,1,94,"182876,25",99,"11,99072",180,350,2,0,3,260,81,66
25499,SERELLO CONSORCIOS ...,94187879,202403,4411,1,140,"138608,62",1,"22,31034",140,79,8,0,0,312,78,1


In [30]:
df['no_cotas_total'] = (df['Quantidade_de_cotas_ativas_em_dia'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_não_contempladas_inadimplentes'].astype(float)  + df['Quantidade_de_cotas_ativas_contempladas_no_mês'].astype(float) + df['Quantidade_de_cotas_excluídas'].astype(float) + df['Quantidade_de_cotas_ativas_quitadas'].astype(float) + df['Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização'].astype(float)).astype(int)

df['qtd_cotas_ativas'] = (df['Quantidade_de_cotas_ativas_em_dia'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_não_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_no_mês'].astype(float)).astype(int)
                          
#   + df['Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização'].astype(float)).astype(int)

# Lista de administradoras que você quer manter o nome original
administradoras_alvo = [
    "ADM CONS NAC HONDA LTDA",
    "BB CONSÓRCIOS",
    "BRADESCO CONS. LTDA.",
    "ITAÚ ADM DE CONSÓRCIOS LTDA",
    "ADM CONS SICREDI LTDA",
    "SICOOB ADM CONS LTDA.",
    "SANTANDER BRASIL ADM CONS LTDA",
    "PORTO SEGURO ADM. CONS. LTDA",
    "EMBRACON ADM CONS S.A.",
    "YAMAHA ADM CONS LTDA",
    "HS ADM CONS LTDA",
    "ADEMICON ADM CONS S.A."
]

# Criando nova coluna com a lógica:
# Se nm_administradora está na lista, mantém o nome
# Caso contrário, retorna "DEMAIS"
df["#adms"] = np.where(
    df["#Nome_da_Administradora"].isin(administradoras_alvo),
    df["#Nome_da_Administradora"],
    "DEMAIS"
)

display(df)


,#Nome_da_Administradora,CNPJ_da_Administradora,Data_base,Código_do_grupo,Código_do_segmento,Número_da_assembléia_geral_ordinária,Valor_médio_do_bem,Índice_de_correção,Taxa_de_administração,Prazo_do_grupo_em_meses,...,Quantidade_de_cotas_ativas_contempladas_inadimplentes,Quantidade_de_cotas_ativas_não_contempladas_inadimplentes,Quantidade_de_cotas_ativas_contempladas_no_mês,Quantidade_de_cotas_excluídas,Quantidade_de_cotas_ativas_quitadas,Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização,no_cotas_total,qtd_cotas_ativas,qtd_faturados,#adms
0,ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202506,20079,3,101,"349580,74",6,"11,96989",100,...,27,0,0,737,873,8,2518,900,19,ITAÚ ADM DE CONSÓRCIOS LTDA
1,ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202506,20179,3,82,"278130,02",99,"11,95467",100,...,142,3,5,739,373,53,2070,905,92,ITAÚ ADM DE CONSÓRCIOS LTDA
2,ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202506,20183,3,82,"48490,07",6,"12,43314",80,...,65,0,0,366,771,4,1983,842,61,ITAÚ ADM DE CONSÓRCIOS LTDA
3,ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202506,20186,3,83,"43432,74",99,"16,63106",80,...,129,0,0,778,818,3,2552,953,126,ITAÚ ADM DE CONSÓRCIOS LTDA
4,ITAÚ ADM DE CONSÓRCIOS LTDA,00000776,202506,20190,3,83,"43107,47",6,"15,90759",80,...,125,0,0,745,822,4,2520,949,121,ITAÚ ADM DE CONSÓRCIOS LTDA
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
213301,SCANIA ADM CONS LTDA.,96479258,202203,03121,2,24,"167958,76",99,"12,92440",100,...,0,28,3,87,1,16,398,294,12,DEMAIS
213302,SCANIA ADM CONS LTDA.,96479258,202203,03122,2,13,"266316,48",99,"13,93446",100,...,0,32,2,27,1,11,308,269,21,DEMAIS
213303,SCANIA ADM CONS LTDA.,96479258,202203,03123,2,3,"309317,57",99,"12,68243",100,...,0,8,1,0,0,3,152,149,5,DEMAIS
213304,SCANIA ADM CONS LTDA.,96479258,202203,03201,2,90,"138624,00",99,"12,44983",100,...,21,4,2,171,136,61,669,301,-36,DEMAIS


In [31]:
df.to_csv(r"C:\Users\joth1\Videos\consórcio\dados_gerais\Imoveis\Bens_Imoveis_Grupos_unificado.csv", sep=';', index=False, encoding='latin1')

In [34]:
### Rodas
df = pd.DataFrame()

lista = [
    r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202506Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202503Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202412Bens_Moveis_Grupos.csv" 
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202409Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202406Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202403Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202312Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202309Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202306Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202303Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202212Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202209Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202206Bens_Moveis_Grupos.csv"
    ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202203Bens_Moveis_Grupos.csv"
    # ,r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\202112Bens_Moveis_Grupos.csv"
]
for i in lista:
    df1 = pd.read_csv(i, sep=';', encoding='latin1', dtype=str)
    df = pd.concat([df, df1], ignore_index=True)

df['no_cotas_total'] = (df['Quantidade_de_cotas_ativas_em_dia'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_não_contempladas_inadimplentes'].astype(float)  + df['Quantidade_de_cotas_ativas_contempladas_no_mês'].astype(float) + df['Quantidade_de_cotas_excluídas'].astype(float) + df['Quantidade_de_cotas_ativas_quitadas'].astype(float) + df['Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização'].astype(float)).astype(int)

df['qtd_cotas_ativas'] = (df['Quantidade_de_cotas_ativas_em_dia'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_não_contempladas_inadimplentes'].astype(float) + df['Quantidade_de_cotas_ativas_contempladas_no_mês'].astype(float)).astype(int)
                          
                          
#   + df['Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização'].astype(float)).astype(int)

df['qtd_faturados'] = (df['Quantidade_de_cotas_ativas_contempladas_inadimplentes'].astype(int) + df['Quantidade_de_cotas_ativas_não_contempladas_inadimplentes'].astype(int)) - df['Quantidade_de_cotas_ativas_com_crédito_pendente_de_utilização'].astype(int)




# Lista de administradoras que você quer manter o nome original
administradoras_alvo = [
    "ADM CONS NAC HONDA LTDA",
    "BB CONSÓRCIOS",
    "BRADESCO CONS. LTDA.",
    "ITAÚ ADM DE CONSÓRCIOS LTDA",
    "ADM CONS SICREDI LTDA",
    "SICOOB ADM CONS LTDA.",
    "SANTANDER BRASIL ADM CONS LTDA",
    "PORTO SEGURO ADM. CONS. LTDA",
    "EMBRACON ADM CONS S.A.",
    "YAMAHA ADM CONS LTDA",
    "HS ADM CONS LTDA",
    "ADEMICON ADM CONS S.A."
]

# Criando nova coluna com a lógica:
# Se nm_administradora está na lista, mantém o nome
# Caso contrário, retorna "DEMAIS"
df["#adms"] = np.where(
    df["#Nome_da_Administradora"].isin(administradoras_alvo),
    df["#Nome_da_Administradora"],
    "DEMAIS"
)


In [35]:
df.to_csv(r"C:\Users\joth1\Videos\consórcio\dados_gerais\Rodas\Bens_Moveis_Grupos_unificado.csv", sep=';', index=False, encoding='latin1')